# RLFT reward types on GEAP

This notebook is a **thin demo**: every step calls into the tested `geap_tuning`
package. It mirrors [`examples/run_rlft_reward_types.py`](../examples/run_rlft_reward_types.py)
and tours the four RLFT reward-scorer shapes the Gen AI SDK exposes, then tunes on
a **composite** of two of them.

RLFT scores each generation with a **reward function** over the record's
`references` — there is no gold completion. The scorer shape decides *how* the
reward is computed:

| Scorer | Builder | When to use |
|---|---|---|
| code-execution | `build_reward_config` | verifiable correctness (ships tested Python) |
| string-match | `build_string_match_reward_config` | cheap format/keyword reward, no gold, no sandbox |
| autorater | `build_autorater_reward_config` | subjective quality via an LLM judge |
| cloud-run | `build_cloud_run_reward_config` | external/custom logic (needs a deployed service) |

A **composite** reward is a weighted sum of single rewards — here verifiable
correctness (0.8) + subjective quality (0.2).

> **Constraints:** RLFT is Pre-GA (`v1beta1`); re-verify scorer symbols before a
> live run. Tuning stays **regional** (never `global`).

> **Requires live GCP and incurs tuning cost.** Have a real `.env` and
> `gcloud auth` in place before the preflight/tune cells.

In [ ]:
from pathlib import Path

from geap_tuning.config import genai_client, load_config

# VERSION parameterizes the display name so reruns reuse the same job (cost control).
BASE_MODEL = "gemini-3.5-flash"
VERSION = "v1"
DISPLAY_NAME = f"geap-rlft-rewards-{VERSION}"
DATA_DIR = Path("datasets/rlft_math")
GCS_PREFIX = "rlft_rewards"
# The autorater scorer REQUIRES an explicit judge model, and the live API only
# accepts a fully-qualified publisher resource path (bare names or
# "publishers/..." fragments fail with an opaque "Internal error occurred for
# computing reward"). Built from cfg below.
AUTORATER_JUDGE = "gemini-2.5-flash"

cfg = load_config()
client = genai_client(cfg)  # tuning is regional-only; global excludes tuning
cfg

## Dataset + one preflight record

Reuse the verifiable-math dataset from the [RLFT notebook](03_rlft.ipynb). Each
record is `contents` + `references` (`{"ground_truth_answer": "<n>"}`), no gold
completion. We keep the first training record to preflight rewards on.

In [ ]:
from geap_tuning.rlft.data import (
    MATH_PROBLEMS,
    build_rlft_dataset,
    build_rlft_records,
    split_dataset,
)

paths = build_rlft_dataset(DATA_DIR)
record = build_rlft_records(split_dataset(MATH_PROBLEMS)[0])[0]
paths

## Build each reward shape

Each builder returns a `SingleReinforcementTuningRewardConfig` with exactly one
scorer sub-object populated. The cloud-run builder is **documented-only** — it is
constructed to show the fourth shape but not preflighted or tuned on (it needs a
separately deployed Cloud Run service).

In [ ]:
from geap_tuning.rlft.tune import (
    build_autorater_reward_config,
    build_cloud_run_reward_config,
    build_composite_reward_config,
    build_reward_config,
    build_string_match_reward_config,
)

code_reward = build_reward_config(reward_name="math_correctness")
string_reward = build_string_match_reward_config()
autorater_model = (
    f"projects/{cfg.project}/locations/{cfg.location}/publishers/google/models/{AUTORATER_JUDGE}"
)
autorater_reward = build_autorater_reward_config(autorater_model=autorater_model)
cloud_run_reward = build_cloud_run_reward_config(
    reward_name="external_scorer",
    cloud_run_uri="https://reward-scorer-xxxxxxxx-uc.a.run.app",
)

# Composite = verifiable correctness (0.8) + subjective quality (0.2).
composite = build_composite_reward_config([(code_reward, 0.8), (autorater_reward, 0.2)])
[c.reward_name for c, _ in [(code_reward, 0), (string_reward, 0), (autorater_reward, 0)]]

## Preflight each reward — no tuning cost

`validate_reward_config` scores one example through a reward and returns the
`overall_reward` / `error`. A non-null error (or `NaN`) means the reward is
broken — fix it before launching (RLFT auto-stops if >80% of reward calls fail).
The composite goes through the `composite_reward_config=` path.

In [ ]:
from geap_tuning.rlft.tune import validate_reward_config

sample_answer = "Let me add them: 2 + 2 = 4.\nAnswer: 4"
for name, single, comp in (
    ("code-execution", code_reward, None),
    ("string-match", string_reward, None),
    ("autorater", autorater_reward, None),
    ("composite", None, composite),
):
    result = validate_reward_config(
        client,
        project=cfg.project,
        location=cfg.location,
        sample_answer=sample_answer,
        example_record=record,
        reward_config=single,
        composite_reward_config=comp,
    )
    print(f"[{name}] {result}")

## Stage data + launch one composite-reward job

Reuse an existing job by display name if present (idempotent reruns), else launch
a fresh RLFT job on the **composite** reward. `reward_config` and
`composite_reward_config` are mutually exclusive — passing the composite skips the
single-reward default.

In [ ]:
from geap_tuning.gcs import upload_file
from geap_tuning.jobs import (
    find_tuning_job_by_display_name,
    tuned_endpoint,
    wait_for_tuning_job,
)
from geap_tuning.rlft.tune import launch_rlft_job

train_uri = upload_file(paths["train"], f"{cfg.bucket}/{GCS_PREFIX}/train.jsonl")
val_uri = upload_file(paths["val"], f"{cfg.bucket}/{GCS_PREFIX}/val.jsonl")

job = find_tuning_job_by_display_name(client, DISPLAY_NAME)
if job is None:
    job = launch_rlft_job(
        client,
        train_uri=train_uri,
        val_uri=val_uri,
        display_name=DISPLAY_NAME,
        base_model=BASE_MODEL,
        composite_reward_config=composite,
        labels=cfg.labels,
    )
job.name

## Wait + evaluate

Wait for completion, resolve the tuned endpoint, and score held-out answers with
the same reward function.

In [ ]:
from geap_tuning.inference import generate
from geap_tuning.rlft.evaluate import run_rlft_eval

job = wait_for_tuning_job(client, job.name)
endpoint = tuned_endpoint(job)

_, _, test_problems = split_dataset(MATH_PROBLEMS)
test_records = build_rlft_records(test_problems)
metrics = run_rlft_eval(
    test_records,
    generate_fn=lambda user_text: generate(client, endpoint, user_text),
)
print(f"Held-out answer accuracy: {metrics['accuracy']:.3f} (n={metrics['n']})")